# Multilingual Farmer Query AI App

## Architecture Overview

This app will enable farmers to ask questions in their regional language and receive answers in the same language.

### Key Components:

1. **Data Ingestion**: Load and process farmer query data
2. **Embedding Generation**: Create vector embeddings for semantic search
3. **Vector Store**: Store embeddings in Databricks Vector Search
4. **RAG Pipeline**: Retrieve relevant context and generate answers
5. **Multilingual Support**: Use models that support regional languages
6. **Offline Capability**: Package as a Databricks App

### Technology Stack:
* **Vector Search**: For similarity-based retrieval
* **Foundation Model APIs**: For multilingual understanding (supports 100+ languages)
* **MLflow**: For model tracking
* **Databricks Apps**: For deployment

In [0]:
%pip install databricks-vectorsearch mlflow langchain langchain-community sentence-transformers gradio --quiet
dbutils.library.restartPython()

In [0]:
import pandas as pd

# Download CSV from Google Drive link and load into a Spark DataFrame
csv_url = "https://drive.google.com/uc?id=1wZ1WI3QE5cgXY4uFEo0szGUpAf8cHc7S&export=download"
pdf = pd.read_csv(csv_url)
df = spark.createDataFrame(pdf)

In [0]:
print(f"Loaded {df.count()} records")
display(df.limit(5))

In [0]:
# Create a catalog and schema for the project
catalog_name = "workspace"  # Use your catalog
schema_name = "farmer_queries"
table_name = "query_data"

# Create schema if it doesn't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

# Save data to Unity Catalog
full_table_name = f"{catalog_name}.{schema_name}.{table_name}"

# Uncomment when data is loaded:
df.write.mode("overwrite").saveAsTable(full_table_name)
print(f"Data saved to {full_table_name}")

print(f"Table will be created at: {full_table_name}")

In [0]:
from databricks.vector_search.client import VectorSearchClient
from pyspark.sql.functions import monotonically_increasing_id

# Initialize Vector Search client
vsc = VectorSearchClient()

# Create vector search endpoint
endpoint_name = "farmer_query_endpoint"
try:
    vsc.create_endpoint(name=endpoint_name)
    print(f"Created endpoint: {endpoint_name}")
except Exception as e:
    print(f"Endpoint may already exist: {e}")

# Enable Change Data Feed on existing table
spark.sql(f"ALTER TABLE {full_table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
print(f"Enabled Change Data Feed on {full_table_name}")

# Add primary key column if it doesn't exist
df_with_id = df.withColumn("id", monotonically_increasing_id())
df_with_id.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name)
print(f"Added 'id' column to {full_table_name}")

# Vector index configuration
index_name = f"{catalog_name}.{schema_name}.query_embeddings_index"

# Create delta sync index with managed embeddings
try:
    vsc.create_delta_sync_index(
        endpoint_name=endpoint_name,
        index_name=index_name,
        source_table_name=full_table_name,
        pipeline_type="TRIGGERED",
        primary_key="id",
        embedding_source_column="questions",  # Column containing the farmer questions
        embedding_model_endpoint_name="databricks-bge-large-en"  # Multilingual model
    )
    print(f"Created vector index: {index_name}")
except Exception as e:
    print(f"Index may already exist: {e}")

## Deployment as Databricks App

To create an offline-capable interface:

### Option 1: Databricks Apps (Recommended)
Create a web interface using Gradio or Streamlit:

```python
import gradio as gr

def interface(query):
    return answer_farmer_query(query)

demo = gr.Interface(
    fn=interface,
    inputs=gr.Textbox(label="Your Question (Any Language)", lines=3),
    outputs=gr.Textbox(label="Answer", lines=5),
    title="Farmer Query Assistant",
    description="Ask questions in your regional language"
)

demo.launch()
```

Save this as `app.py` and deploy using Databricks Apps.

### Option 2: Model Serving Endpoint
Deploy the RAG function as an MLflow model for API access:
1. Log the model with MLflow
2. Register to Unity Catalog
3. Create a Model Serving endpoint
4. Build a mobile/web app that calls the endpoint

### Offline Capability:
* **Online mode**: Uses Databricks compute and models
* **True offline**: Would require local model deployment (edge device)
* **Hybrid**: Cache common queries locally, sync when online

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.vector_search.client import VectorSearchClient

w = WorkspaceClient()
vsc = VectorSearchClient(
    workspace_url=w.config.host,
    personal_access_token=w.config.token,
    disable_notice=True
)

In [0]:
!pip show databricks-sdk

In [0]:
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

# List service principals and find yours
for sp in w.service_principals.list():
    if "farmer" in sp.display_name.lower() or "55f1wk" in sp.display_name.lower():
        print(f"Display name: {sp.display_name}")
        print(f"Application ID: {sp.application_id}")

In [0]:
app_id = "8e7c8ddc-ffc6-47d5-be3a-0b907eb4a349"  # replace with the number from above

spark.sql(f"GRANT USE CATALOG ON CATALOG workspace TO `{app_id}`")
spark.sql(f"GRANT USE SCHEMA ON SCHEMA workspace.farmer_queries TO `{app_id}`")
spark.sql(f"GRANT SELECT ON TABLE workspace.farmer_queries.query_embeddings_index TO `{app_id}`")

In [0]:
endpoint = w.serving_endpoints.get(name="qwen_dense_model")
print(endpoint.id)

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ServingEndpointPermissionLevel, ServingEndpointAccessControlRequest

w = WorkspaceClient()

w.serving_endpoints.set_permissions(
    serving_endpoint_id="822e0472b3534e34a704813364784040",
    access_control_list=[
        ServingEndpointAccessControlRequest(
            service_principal_name="8e7c8ddc-ffc6-47d5-be3a-0b907eb4a349",
            permission_level=ServingEndpointPermissionLevel.CAN_QUERY
        )
    ]
)

In [0]:
import mlflow
from mlflow.tracking import MlflowClient

client = MlflowClient()

versions = client.search_model_versions("name='workspace.farmer_queries.qwen_dense_model'")
for v in versions:
    print(f"Version: {v.version}")
    model_info = mlflow.models.get_model_info(f"models:/workspace.farmer_queries.qwen_dense_model/{v.version}")
    print(model_info.signature)

In [0]:
versions